# Dust Percentage Analysis by Month

This notebook analyzes the `DustSCAN_2022.nc` dataset to identify the months where the dust percentage is the highest.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Path to the dataset
file_path = os.path.expanduser('~/Downloads/DustSCAN_2022.nc')

# Load the dataset
try:
    ds = xr.open_dataset(file_path, engine='netcdf4')
    print("Dataset loaded successfully.")
    print(ds)
except Exception as e:
    print(f"Error loading dataset: {e}")

In [ ]:
# Group by month and calculate the percentage of dust pixels
# Dust is present where plume_id > 0

# Create a boolean mask for dust presence
is_dust = ds['plume_id'] > 0

# Group by month ('time.month') and calculate the mean
# The mean of a boolean array gives the percentage of True values
monthly_dust_fraction = is_dust.groupby('time.month').mean(dim=['time', 'lat', 'lon'])

# Convert fraction to percentage
monthly_dust_percentage = monthly_dust_fraction * 100

print("Monthly Dust Percentage:")
print(monthly_dust_percentage.values)

In [ ]:
# Convert the result to a pandas DataFrame for easier plotting and tabular view
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
available_months = [months[i-1] for i in monthly_dust_percentage.month.values]

df_dust = pd.DataFrame({
    'Month': available_months,
    'Dust Percentage (%)': monthly_dust_percentage.values
})

# Sort by dust percentage descending to see the highest months first
df_sorted = df_dust.sort_values(by='Dust Percentage (%)', ascending=False).reset_index(drop=True)
display(df_sorted)

In [ ]:
# Plotting the results
plt.figure(figsize=(10, 6))
plt.bar(df_dust['Month'], df_dust['Dust Percentage (%)'], color='coral', edgecolor='black')
plt.title('Average Dust Percentage by Month (2022)', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Dust Percentage (%)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i, val in enumerate(df_dust['Dust Percentage (%)']):
    plt.text(i, val + 0.05, f'{val:.2f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

### Conclusion
Based on the analysis, we can identify which months exhibit the highest dust activity according to the `plume_id` mask in the dataset.